# Module 2 — Agentic RAG: A LangGraph Retrieval Loop

**The problem, from Module 1:** one-shot RAG — embed the question, vector-search once, one LLM
call — works well on single-chunk factual questions but falls apart on questions that need
*multiple, targeted* retrievals across documents. A single top-k=5 vector search over "3M vs
Apple R&D" mostly returns 3M chunks (they happen to score higher) and the model has no way to
go back and specifically look for Apple's numbers.

**What we build in this module:** an agentic retrieval loop with LangGraph that can decide what
to search for, evaluate whether it has enough, go back for more, generate an answer from a
curated knowledge base, and critique its own answer before returning it.

**New components introduced:**
- `retrieval.vector` — semantic search over `Chunk.embedding` (rewritten for Module 2: company
  filtering, tool-shaped return values)
- `retrieval.keyword` — fulltext (Lucene) search over the `chunk_text` index
- `retrieval.graph_nav` — page-range lookup for a known `doc_id`
- `agent.tools` — the three retrieval functions wrapped as LangChain `@tool`s
- `agent.state` — `AgentState` (the graph's shared state) and the two structured grading
  schemas, `RetrievalGrade` and `AnswerGrade`
- `agent.prompts` — prompt builders for each LLM-backed node
- `agent.nodes` — the five node functions (each callable standalone, for inspection)
- `agent.graph` — `build_agent()`, wiring the nodes into a compiled LangGraph `StateGraph`

## 1. Recap — Where the One-Shot Baseline Struggles

Same question that closed out Module 1. Neo4j already holds both filings from that module's
ingestion run, so we can call straight into `qa.baseline.ask` and watch it fail the same way.

In [ ]:
from financial_advisor.qa.baseline import ask


In [ ]:

HARD_QUESTIONS = (
    "Compare 3M's and Apple's approach to research and development investment, "
        "based on their 2025 10-Ks.",
    "How did the number of 3M's U.S. manufacturing facilities change between the 2024 "
        "and 2025 10-K filings?",
)

for q in HARD_QUESTIONS:
    print(f"\n{'=' * 80}")
    print(f"Q: {q}\n")
    print(f"A: {ask(q, k=5)}")

## 2. Three Retrieval Tools

The agent gets three tools instead of a single fixed vector search, each suited to a different
retrieval situation. They're plain LangChain `@tool`-decorated functions in `agent.tools`, thin
wrappers around `retrieval.vector` / `retrieval.keyword` / `retrieval.graph_nav`:

| Tool | Backing index | Best for |
|---|---|---|
| `semantic_search` | `chunk_embedding` (vector) | broad, conceptual, or comparative questions; unsure of exact wording |
| `fulltext_search` | `chunk_text` (Lucene fulltext) | exact terminology, line items, section titles |
| `get_document_pages` | none — direct `Chunk.doc_id` match | pulling full context once a `doc_id` is already known |

`semantic_search` and `fulltext_search` both accept optional `company_id` (`"3M"` / `"APPLE"`)
and `year` (`2024` / `2025`) filters — critical once a company has more than one filing in the
corpus, since without `year` a search has no way to isolate one 10-K from the other; both years'
chunks compete equally. (Section 2b below covers how `year` filtering is wired into the graph —
it isn't there by default, we add it.) Every returned chunk carries its `doc_id`, so a
`semantic_search` or `fulltext_search` hit is what feeds a later `get_document_pages` call.

Let's call each one directly, exactly as the agent will.

In [ ]:
from financial_advisor.agent.tools import fulltext_search, get_document_pages, semantic_search

# Tools are LangChain @tool objects — .invoke(dict) runs them exactly as the agent will call them
hits = semantic_search.invoke(
    {"query": "3M research and development expenses 2025", "k": 3, "company_id": "3M"}
)
print(f"semantic_search -> {len(hits)} chunk(s)\n")
for h in hits:
    print(f"  [{h['score']:.3f}] {h['doc_id']}  chunk_id={h['id']}  pages={h['pages']}")
    print(f"    {h['text'][:160]}\n")

In [ ]:
text_hits = fulltext_search.invoke(
    {"query": '"research and development"', "k": 3, "company_id": "APPLE"}
)
print(f"fulltext_search -> {len(text_hits)} chunk(s)\n")
for h in text_hits:
    print(f"  [{h['score']:.3f}] {h['doc_id']}  chunk_id={h['id']}  pages={h['pages']}")
    print(f"    {h['text'][:160]}\n")

In [ ]:
# Given a doc_id already surfaced by one of the tools above, pull specific pages directly
doc_id = hits[0]["doc_id"]
target_pages = hits[0]["pages"]
page_hits = get_document_pages.invoke({"doc_id": doc_id, "pages": target_pages, "limit": 5})

print(f"get_document_pages({doc_id!r}, pages={target_pages}) -> {len(page_hits)} chunk(s)\n")
for h in page_hits:
    print(f"  {h['doc_id']}  chunk_id={h['id']}  pages={h['pages']}")
    print(f"    {h['text'][:160]}\n")

## 2b. Graph Model Refactoring — Adding `year`

The `company_id` filter above isn't enough on its own: 3M and Apple each have *two* filings in
this corpus (2024, 2025). A `semantic_search` scoped to `company_id="3M"` searches across both
years at once — for a question about one specific filing, both years' chunks compete for the
same top-k budget, with nothing telling the agent which year a given chunk came from. This is
exactly what tripped up the manufacturing-facilities question earlier.

The reason `year` isn't already filterable: it lives on `Document`, not `Chunk` — the level
`semantic_search`/`fulltext_search` actually operate at. This is a small, real graph-model
change: denormalize `Document.year` onto `Chunk.year` (the same idea as `company_id`, which was
already denormalized this way back in Module 1).

Neo4j 2026.01+ lets a vector index declare **additional properties** for in-index pre-filtering
— `CREATE VECTOR INDEX ... WITH [n.prop1, n.prop2] ...`, queried with the Cypher 25 `SEARCH ...
VECTOR INDEX ... WHERE ...` clause. That replaces `semantic_search`'s old `company_id`-only,
over-fetch-then-filter-in-Cypher approach with an exact in-index filter, and folds `year` in for
free. Full-text (Lucene) indexes have no equivalent — `chunk_text` keeps the over-fetch-then-
`WHERE` pattern, just extended to cover `year` too.

`ingestion.schema.apply_agentic_schema()` does both steps: backfills `Chunk.year` for any chunk
that doesn't have it yet, then rebuilds `chunk_embedding` with the additional properties — it
checks the index's current `properties` first and skips the rebuild if already done, so it's
safe to re-run.

In [ ]:
from financial_advisor.ingestion.schema import apply_agentic_schema
from financial_advisor.services.embedding_service import embedding_service
from financial_advisor.services.neo4j_service import neo4j_service

apply_agentic_schema(embedding_service.dimensions)

rows = neo4j_service.run_query(
    "SHOW INDEXES YIELD name, type, properties WHERE name = 'chunk_embedding' "
    "RETURN name, type, properties"
)
for r in rows:
    print(r)

In [ ]:
# Same tool, same company — now scoped to one filing year at a time
for yr in (2024, 2025):
    hits = semantic_search.invoke(
        {"query": "manufacturing facilities properties", "k": 2, "company_id": "3M", "year": yr}
    )
    print(f"year={yr}:")
    for h in hits:
        print(f"  [{h['score']:.3f}] {h['doc_id']}  year={h['year']}")
        print(f"    {h['text'][:160]}\n")

## 3. Shared State

LangGraph nodes read and write a single shared `AgentState` (a `TypedDict`, defined in
`agent.state`). Each node returns only the keys it changes; LangGraph merges them into the
running state.

Two fields are worth calling out because they carry the whole design:

- **`growing_knowledge`** (str) — a self-contained, cumulative write-up of every fact needed to
  answer the question, rebuilt each retrieval round by the grading node. This — *not* the raw
  retrieved chunks — is all the answer generator ever sees. Retrieved chunks can be noisy,
  redundant, or only partially relevant; forcing an LLM to distill them into
  `growing_knowledge` first keeps the final answering step focused and cheap.
- **`retrieved_chunks`** (list[dict]) — every unique chunk pulled back so far, deduplicated by
  `id` and accumulated *across* retrieval rounds, so a second round doesn't lose what the first
  one found.

Two Pydantic models drive the two evaluation nodes via `with_structured_output`:

- **`RetrievalGrade`** — `sufficient: bool`, the rewritten `growing_knowledge: str`, and
  `feedback: str` for the next retrieval round.
- **`AnswerGrade`** — `accepted: bool`, `next_action: Literal["retry_answer",
  "retry_retrieval", "end"]`, and `feedback: str`.

`next_action` is the crux of the answer-evaluation step: the grading LLM has to look at
`growing_knowledge` and decide *why* the answer is wrong — data missing (go back to retrieval)
versus data present but misused (go back to answer generation).

In [ ]:
from financial_advisor.agent.state import initial_state

state = initial_state(HARD_QUESTIONS[0])
for k, v in state.items():
    print(f"  {k}: {v!r}")

## 4. The Graph

Five nodes, two LLM-graded decision points. The happy path runs top to bottom once; each side
loop is one place the agent can decide it isn't done yet.

```
                              ┌────────────┐
                              │   START    │
                              └─────┬──────┘
                                    │
                                    ▼
                     ┌──────────────────────────────┐
              ┌─────▶│  1. retriever_strategy_agent   │
              │      │     picks tool(s) to call,     │
              │      │     given feedback so far       │
              │      └───────────────┬────────────────┘
              │                      │ tool_calls
              │                      ▼
              │      ┌──────────────────────────────┐
              │      │  2. call_tools                 │
              │      │     runs semantic_search /      │
              │      │     fulltext_search /           │
              │      │     get_document_pages          │
              │      └───────────────┬────────────────┘
              │                      │ retrieved_chunks (accumulated)
              │                      ▼
              │      ┌──────────────────────────────┐
              │      │  3. evaluate_retrieval          │
              └──────┤     rewrites growing_knowledge, │
               retry │     decides: sufficient?         │
                      └───────────────┬────────────────┘
                                      │ continue (sufficient, or iteration cap)
                                      ▼
                     ┌──────────────────────────────┐
              ┌─────▶│  4. generate_answer            │
              │      │     sees ONLY                   │
              │      │     growing_knowledge —         │
              │      │     no raw chunks               │
              │      └───────────────┬────────────────┘
       retry_answer                  │ answer
   (info was there,                  ▼
    answer missed it)  ┌──────────────────────────────┐
              └────────┤  5. evaluate_answer             │
                        │     correct & complete?         │
                        └──────┬───────────────┬─────────┘
                               │               │
                          end  │               │ retry_retrieval
                               ▼               │ (info genuinely missing)
                            ┌─────┐             │
                            │ END │             └──────────▶ back to node 1
                            └─────┘
```

Three separate loops, each with a different trigger:
1. **`evaluate_retrieval` → `retriever_strategy_agent`** ("retry") — this round's chunks (plus
   everything before them) aren't enough yet; go search again.
2. **`evaluate_answer` → `generate_answer`** ("retry_answer") — the knowledge is there, the
   answer just didn't use it correctly.
3. **`evaluate_answer` → `retriever_strategy_agent`** ("retry_retrieval") — writing the answer
   exposed a gap that grading the retrieval missed; go search again.

Both retrieval and answering have hard iteration caps (`MAX_RETRIEVAL_ITERATIONS`,
`MAX_ANSWER_ATTEMPTS` in `agent.nodes`) so a stubborn question degrades to "best effort" instead
of looping forever.

In [ ]:
from financial_advisor.agent.graph import build_agent
from IPython.display import Image, display

agent = build_agent()

# Cross-check the hand-drawn diagram above against LangGraph's own view of the wiring

display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

## 5. Run It on the Question That Broke Module 1

Every node prints its own progress (see `agent.nodes`), so invoking the compiled graph gives us
a live trace of the loop: which tools got called, whether retrieval was graded sufficient, and
whether the answer was accepted.

In [ ]:
result = agent.invoke(initial_state(HARD_QUESTIONS[0]), {"recursion_limit": 50})

print(f"\nretrieval_iterations: {result['retrieval_iterations']}")
print(f"answer_attempts:      {result['answer_attempts']}")
print(f"chunks retrieved:     {len(result['retrieved_chunks'])}")
print(f"\n{'=' * 80}\nANSWER:\n{'=' * 80}\n{result['answer']}")

Compare that against the Module 1 baseline's answer at the top of this notebook: same question,
same underlying data, but here the strategy agent can issue a `semantic_search` scoped to each
`company_id` independently — so both companies' numbers make it into `growing_knowledge` — and
`evaluate_answer` would have looped back to retrieval had one side been missing.

`growing_knowledge` is what actually reached `generate_answer` — the answer generator never saw
`retrieved_chunks` directly:

In [ ]:
print(result["growing_knowledge"])

## 6. A Few More Questions — Easy vs. Harder

The single-chunk factual questions Module 1 already handled well should still resolve in one
retrieval round here — the agentic loop shouldn't cost extra iterations when it doesn't need
them. The harder case below is one Module 1's baseline got wrong: 3M's U.S. manufacturing
facility count is reported once per filing year (`Item 2. Properties`), not in a joint
year-over-year table, so both years' counts had to compete for one shared top-5 budget and
one of them lost. Here the strategy agent can scope a search to each filing independently and
run a second round if the first wasn't enough.

In [ ]:
EXAMPLE_QUESTIONS = [
    "What is 3M's principal executive office address?",  # single-chunk factual — easy
    "What was Apple's total revenue in fiscal year 2024?",  # single-chunk factual — easy
    "How did the number of 3M's U.S. manufacturing facilities change between the 2024 "
    "and 2025 10-K filings?",  # Module 1's baseline said "I don't know" here — hard
]

for q in EXAMPLE_QUESTIONS:
    print(f"\n{'=' * 80}\nQ: {q}\n{'=' * 80}")
    r = agent.invoke(initial_state(q), {"recursion_limit": 50})
    print(
        f"\n[{r['retrieval_iterations']} retrieval round(s), "
        f"{r['answer_attempts']} answer attempt(s), "
        f"{len(r['retrieved_chunks'])} chunk(s) retrieved]"
    )
    print(f"\nA: {r['answer']}")

## 7. Where This Still Struggles

The loop is only as good as the graph it searches. `Company → Document → Chunk` is text sliced
out of 10-Ks — it has no notion of *people* and how they connect across companies. 3M's 10-K
does list its executive officers, but for their biographical history it explicitly defers to a
document we never ingested: 3M's proxy statement. Watch what happens when we ask for exactly
that.

In [ ]:
STRUGGLE_QUESTION = (
    "Which other companies have 3M's current executive officers previously worked at or "
    "served as directors of?"
)

r = agent.invoke(initial_state(STRUGGLE_QUESTION), {"recursion_limit": 50})
print(
    f"\n[{r['retrieval_iterations']} retrieval round(s), "
    f"{r['answer_attempts']} answer attempt(s), "
    f"{len(r['retrieved_chunks'])} chunk(s) retrieved]"
)
print(f"\nA: {r['answer']}")

With `year` filtering in place (section 2b), the strategy agent actually finds more than it
used to here: 3M's 10-K Item 1 has an "Executive Officers of the Registrant" table listing each
officer's *other positions held during 2020-2024* — e.g. William M. Brown at L3Harris
Technologies, Anurag Maheshwari at Otis Worldwide. Once retrieval can scope precisely to
`company_id="3M", year=2024`, that table gets surfaced and cited in the answer, instead of
getting lost in an unscoped search.

But read what the answer itself says about its own limits: that table only covers a five-year
window, and explicitly defers to 3M's proxy statement — never ingested — for full director
biographies and anything earlier. `get_executives` (bound to the agent from the start) still
returns nothing, since Module 3's `Person`/`ROLE_AT` data doesn't exist yet. Better retrieval
surfaces more of what the corpus *actually contains*; it can't manufacture what isn't there.
That's the real gap Module 3 closes — not "the agent finds nothing" but "the agent finds a
caveated, time-boxed fragment, and a dedicated data source removes the caveats."

## 8. What's Next

Module 3 enriches the graph with `Person`, `Event`, and `Article` nodes (`ROLE_AT`,
`MENTIONED_IN`) — pulling in exactly the executive-history and news data the previous section's
question needed. The retrieval loop built here stays as-is; it just gets more of the graph, and
more tools, to work with.